In [ ]:
# install packages - did this in command line conda environment
# remotes::install_github("jokergoo/circlize@9b21578")
# remotes::install_github("jokergoo/ComplexHeatmap@7d95ca5")
# remotes::install_github("immunogenomics/presto@31dc97f")
# remotes::install_github("jinworks/CellChat@88c2e13")
# BiocManager::install("GenomeInfoDbData")

In [1]:
suppressPackageStartupMessages({
    library(tidyverse)
    library(zellkonverter)
    # library(scater)
    #library(scran)
    # library(scuttle)
    library(SingleCellExperiment)
    library(CellChat)
})

In [2]:
dist_out_dir <- "/home/workspace/spatial_mouse_lung_outputs/downstream_analysis/distance"

cellchat_out_dir <- file.path(dist_out_dir, "cellchat")
if (!dir.exists(cellchat_out_dir)) {
  dir.create(cellchat_out_dir, recursive = TRUE)
}



In [ ]:
sce = readH5AD(file.path(dist_out_dir, "adata_cellchat_prepped.h5ad"))

Warning message:
“The names of these selected uns items have been modified to match R
conventions: '_scvi_manager_uuid' -> 'X_scvi_manager_uuid' and '_scvi_uuid' ->
'X_scvi_uuid'”
Warning message:
“The names of these selected obs columns have been modified to match R
conventions: '_scvi_batch' -> 'X_scvi_batch' and '_scvi_labels' ->
'X_scvi_labels'”


In [ ]:
#sce$Timepoint <- stringr::str_extract(sce$batch, "\\d+")
sce$sample_label <- factor(sce$sample_label, levels = c("HDM_day3", "HDM_day30"))

In [ ]:
reducedDimNames(sce)

In [ ]:
run_cellchat <- function(sce_tmp, name, out_dir) {
    
    print("step: data.input"); flush.console()
    data.input = assay(sce_tmp, "X") # X are the log norm counts here, see part 1
    meta = data.frame(labels = sce_tmp$label_fine,
                    samples = sce_tmp$sample_label_cp, # (KA - I only have one sample, this is just a copy of sample_label
                    row.names = colnames(sce_tmp))
    print("step: spatial.locs"); flush.console()
    # spatial.locs = reducedDim(sce_tmp, 'spatial') |> as.data.frame() # KA this gives me an error 
    # spatial.locs = as.matrix(reducedDim(sce_tmp, 'spatial')) # this takes a long time
    spatial.locs = reducedDim(sce_tmp, 'spatial')  # try this
    scale.factors = list(spot.diameter = 5, spot = 5)
    spatial.factors = data.frame(ratio = 1, tol = 5)

    print("running: createCellChat")
    cellchat <-
        createCellChat(
            object = data.input,
            meta = meta,
            group.by = "labels",
            datatype = "spatial",
            coordinates = spatial.locs,
            spatial.factors = spatial.factors
        )


    CellChatDB <- CellChatDB.mouse # use CellChatDB.human if running on human data

    # use a subset of CellChatDB for cell-cell communication analysis
    # CellChatDB.use <- subsetDB(CellChatDB, search = "Secreted Signaling") # use Secreted Signaling
    # use all CellChatDB for cell-cell communication analysis
    CellChatDB.use <- CellChatDB # simply use the default CellChatDB

    # set the used database in the object
    cellchat@DB <- CellChatDB.use

    # subset the expression data of signaling genes for saving computation cost
    cellchat <- subsetData(cellchat) # This step is necessary even if using the whole database

    # future::plan("multisession", workers = 8) # do parallel
    print("running: identifyOverExpressedGenes, identifyOverExpressedInteractions")
    cellchat <- identifyOverExpressedGenes(cellchat)
    cellchat <- identifyOverExpressedInteractions(cellchat)

    # Typically, contact.range = 10, which is a typical human cell size
    print("running: computeCommunProb")
    cellchat <- computeCommunProb(cellchat,
        # type = "truncatedMean", trim = 0.1, # try 0.1, could lower to 0.05
        type = "truncatedMean", trim = 0.05, # try 0.1, could lower to 0.05
        distance.use = TRUE, interaction.range = 100,
        scale.distance = 1,
        contact.dependent = TRUE, contact.range = 10
    )
    # Filter out the cell-cell communication if there are only few number of cells in certain cell groups
    print("running: filterCommunication")
    cellchat <- filterCommunication(cellchat, min.cells = 10)

    print("running: computeCommunProbPathway")
    cellchat <- computeCommunProbPathway(cellchat)
    cellchat <- aggregateNet(cellchat)

    print(paste("running: saveRDS for sample: ", name))
    # saveRDS(cellchat, file = file.path(out_dir, paste0("cellchat_",name,".rds")))
    saveRDS(cellchat, file = file.path(out_dir, paste0("cellchat_trim05_",name,".rds")))
}

In [ ]:
cellchat_sample <- function(sce, sample_label_select, out_dir) {

    sce_tmp = sce[,sce$sample_label == sample_label_select]

    # # try on subset
    # sce_tmp <- sce_tmp[, sample(1:ncol(sce_tmp), 1000)]
    
    # This is absolutely key. Otherwise Cellchat does not WORK!
    # KA note here - we have only one sample per condition - make a copy of the sample_label column 
    sce_tmp$sample_label_cp <- sce_tmp$sample_label
    sce_tmp$sample_label_cp <- droplevels(sce_tmp$sample_label_cp)
    # KA also do this for label column
    sce_tmp$label_fine <- droplevels(sce_tmp$label_fine)

    print("run_cellchat"); flush.console()
    run_cellchat(sce_tmp, sample_label_select, out_dir)
}

In [ ]:
cellchat_out_dir

In [ ]:
cellchat_sample(sce, 'HDM_day3', cellchat_out_dir)
cellchat_sample(sce, 'HDM_day30', cellchat_out_dir)


In [ ]:
sce